This notebook performs:

- Probe ID → Gene Symbol conversion (Microarray: AD, MS)
- ENSEMBL → Gene Symbol conversion (RNA-Seq: PD, HD)
- Duplicate removal via gene-level aggregation
- Saving harmonized datasets

**3. ID Harmonization (gene symbols)**

AD and MS datasets (microarray)

probe IDs -> Gene Symbols


Imports

In [ ]:
import pandas as pd


Load data + GPL

In [ ]:
# Load expression matrices
ad = pd.read_csv("AD_expression_matrix.csv", index_col=0)
ms = pd.read_csv("MS_expression_matrix.csv", index_col=0)

# Load GPL570 annotation
gpl = pd.read_csv("GPL570-55999.txt", sep="\t", comment="#")

# Keep only probe ID and Gene Symbol
gpl = gpl[['ID', 'Gene Symbol']].dropna()
gpl = gpl[gpl['Gene Symbol'] != '']
gpl = gpl.rename(columns={'ID': 'probe_id', 'Gene Symbol': 'gene_symbol'})

# Create mapping
probe_to_gene = dict(zip(gpl['probe_id'], gpl['gene_symbol']))


/tmp/ipython-input-4040058444.py:6: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  gpl = pd.read_csv("GPL570-55999.txt", sep="\t", comment="#")


In [ ]:
#AD mapping
ad['gene_symbol'] = ad.index.map(probe_to_gene)
ad = ad.dropna(subset=['gene_symbol'])
ad = ad.groupby('gene_symbol').mean()


In [ ]:
#MS mapping
ms['gene_symbol'] = ms.index.map(probe_to_gene)
ms = ms.dropna(subset=['gene_symbol'])
ms = ms.groupby('gene_symbol').mean()


In [ ]:
#Save ad/ms
ad.to_csv("AD_geneSymbol.csv")
ms.to_csv("MS_geneSymbol.csv")

PD and HD datasets (affymetrix)

ENSMBL IDs -> Gene Symbols



In [ ]:
!pip install mygene


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.2 MB/s eta 0:00:00


In [ ]:
import mygene
mg = mygene.MyGeneInfo()


In [ ]:
def ensembl_to_symbol(df):
    genes = df.index.str.split('.').str[0]  # remove version numbers
    res = mg.querymany(
        genes.tolist(),
        scopes="ensembl.gene",
        fields="symbol",
        species="human"
    )
    mapping = {r['query']: r.get('symbol') for r in res if 'symbol' in r}
    df['gene_symbol'] = genes.map(mapping)
    df = df.dropna(subset=['gene_symbol'])
    return df.groupby('gene_symbol').mean()


In [ ]:
hd = pd.read_csv("HD_expression_matrix.csv", index_col=0)
hd = ensembl_to_symbol(hd)


INFO:biothings.client:querying 1-1000 ...
INFO:biothings.client:querying 1001-2000 ...
INFO:biothings.client:querying 2001-3000 ...
INFO:biothings.client:querying 3001-4000 ...
INFO:biothings.client:querying 4001-5000 ...
INFO:biothings.client:querying 5001-6000 ...
INFO:biothings.client:querying 6001-7000 ...
INFO:biothings.client:querying 7001-8000 ...
INFO:biothings.client:querying 8001-9000 ...
INFO:biothings.client:querying 9001-10000 ...
INFO:biothings.client:querying 10001-11000 ...
INFO:biothings.client:querying 11001-12000 ...
INFO:biothings.client:querying 12001-13000 ...
INFO:biothings.client:querying 13001-14000 ...
INFO:biothings.client:querying 14001-15000 ...
INFO:biothings.client:querying 15001-16000 ...
INFO:biothings.client:querying 16001-17000 ...
INFO:biothings.client:querying 17001-18000 ...
INFO:biothings.client:querying 18001-19000 ...
INFO:biothings.client:querying 19001-20000 ...
INFO:biothings.client:querying 20001-21000 ...
INFO:biothings.client:querying 2100

In [ ]:
def clean_gene_symbols(df):
    df = df.copy()
    df.index = (
        df.index
        .astype(str)
        .str.upper()  # fix case
        .str.strip()  # remove spaces
        .str.split("///")  # handle Affy multi-mapping
        .str[0]
    )
    df = df[~df.index.isna()]
    df = df[df.index != ""]
    return df

In [ ]:
pd_ds.index = pd_ds.index.str.split('.').str[0]
hd.index = hd.index.str.split('.').str[0]

In [ ]:
pd_raw.index = pd_raw.index.str.split('.').str[0]

In [ ]:
import mygene
import pandas as pd

mg = mygene.MyGeneInfo()

def enst_to_symbol(df):
    """
    Convert ENSEMBL transcript IDs (ENST) to gene symbols
    """
    enst_ids = df.index.tolist()

    query = mg.querymany(
        enst_ids,
        scopes="ensembl.transcript",
        fields="symbol",
        species="human"
    )

    mapping = {}
    for item in query:
        if "query" in item and "symbol" in item:
            mapping[item["query"]] = item["symbol"]

    df = df[df.index.isin(mapping.keys())]
    df.index = df.index.map(mapping)
    df.index.name = "gene_symbol"

    return df


In [ ]:
pd_ds = enst_to_symbol(pd_raw)


INFO:biothings.client:querying 1-1000 ...
INFO:biothings.client:querying 1001-2000 ...
INFO:biothings.client:querying 2001-3000 ...
INFO:biothings.client:querying 3001-4000 ...
INFO:biothings.client:querying 4001-5000 ...
INFO:biothings.client:querying 5001-6000 ...
INFO:biothings.client:querying 6001-7000 ...
INFO:biothings.client:querying 7001-8000 ...
INFO:biothings.client:querying 8001-9000 ...
INFO:biothings.client:querying 9001-10000 ...
INFO:biothings.client:querying 10001-11000 ...
INFO:biothings.client:querying 11001-12000 ...
INFO:biothings.client:querying 12001-13000 ...
INFO:biothings.client:querying 13001-14000 ...
INFO:biothings.client:querying 14001-15000 ...
INFO:biothings.client:querying 15001-16000 ...
INFO:biothings.client:querying 16001-17000 ...
INFO:biothings.client:querying 17001-18000 ...
INFO:biothings.client:querying 18001-19000 ...
INFO:biothings.client:querying 19001-20000 ...
INFO:biothings.client:querying 20001-21000 ...
INFO:biothings.client:querying 2100

In [ ]:
pd_ds.index[:10]

Index(['ARF5', 'M6PR', 'ESRRA', 'FKBP4', 'CYP26B1', 'NDUFAF7', 'FUCA2',
       'DBNDD1', 'HS3ST1', 'SEMA3F'],
      dtype='object', name='gene_symbol')

In [ ]:
pd_ds.to_csv("PD_geneSymbol.csv")
hd.to_csv("HD_geneSymbol.csv")


**4. DUPLICATE GENE REMOVAL**

In [ ]:
def remove_duplicate_genes(df):
    # drop rows with missing or empty gene symbols
    df = df[~df.index.isna()]
    df = df[df.index != ""]

    # average duplicates
    df = df.groupby(df.index).mean()
    return df


In [ ]:
ad = remove_duplicate_genes(ad)
ms = remove_duplicate_genes(ms)
pd_ds = remove_duplicate_genes(pd_ds)
hd = remove_duplicate_genes(hd)


In [ ]:
ad = remove_duplicate_genes(ad)
ms = remove_duplicate_genes(ms)
pd_ds = remove_duplicate_genes(pd_ds)
hd = remove_duplicate_genes(hd)


In [ ]:
print("AD:", ad.shape)
print("MS:", ms.shape)
print("PD:", pd_ds.shape)
print("HD:", hd.shape)


AD: (23520, 173)
MS: (23520, 29)
PD: (31508, 16)
HD: (23118, 69)


**5. COMMON GENE INTERSECTION**

In [ ]:
common_genes = (
    set(ad.index)
    & set(ms.index)
    & set(pd_ds.index)
    & set(hd.index)
)

print("Number of common genes:", len(common_genes))


Number of common genes: 14784


In [ ]:
common_genes = sorted(common_genes)

ad_c = ad.loc[common_genes]
ms_c = ms.loc[common_genes]
pd_c = pd_ds.loc[common_genes]
hd_c = hd.loc[common_genes]


In [ ]:
print(ad_c.shape, ms_c.shape, pd_c.shape, hd_c.shape)

(14784, 173) (14784, 29) (14784, 16) (14784, 69)


In [ ]:
ad_c.to_csv("AD_commonGenes.csv")
ms_c.to_csv("MS_commonGenes.csv")
pd_c.to_csv("PD_commonGenes.csv")
hd_c.to_csv("HD_commonGenes.csv")


In [ ]:
pd.Series(common_genes, name="geneSymbol").to_csv(
    "COMMON_GENES_14784.csv", index=False
)

**6. z- Score STANDARDIZATION**

In [ ]:
ad = pd.read_csv("AD_commonGenes.csv", index_col=0)
ms = pd.read_csv("MS_commonGenes.csv", index_col=0)
pd_ds = pd.read_csv("PD_commonGenes.csv", index_col=0)
hd = pd.read_csv("HD_commonGenes.csv", index_col=0)

print(ad.shape, ms.shape, pd_ds.shape, hd.shape)

(14784, 173) (14784, 29) (14784, 16) (14784, 69)


In [ ]:
from scipy.stats import zscore
import pandas as pd

ad_z = pd.DataFrame(
    zscore(ad, axis=1, nan_policy='omit'),
    index=ad.index,
    columns=ad.columns
)

ms_z = pd.DataFrame(
    zscore(ms, axis=1, nan_policy='omit'),
    index=ms.index,
    columns=ms.columns
)

pd_z = pd.DataFrame(
    zscore(pd_ds, axis=1, nan_policy='omit'),
    index=pd_ds.index,
    columns=pd_ds.columns
)

hd_z = pd.DataFrame(
    zscore(hd, axis=1, nan_policy='omit'),
    index=hd.index,
    columns=hd.columns
)


In [ ]:
ad_z.mean(axis=1).head()
ad_z.std(axis=1).head()

,0
gene_symbol,
A1BG,1.002903
A1CF,1.002903
A2M,1.002903
A2ML1,1.002903
A2MP1,1.002903


In [ ]:
ad_z.to_csv("AD_zscore.csv")
ms_z.to_csv("MS_zscore.csv")
pd_z.to_csv("PD_zscore.csv")
hd_z.to_csv("HD_zscore.csv")


In [ ]:
ad_z.std(axis=1).describe()

,0
count,1.478400e+04
mean,1.002903e+00
std,3.568780e-13
min,1.002903e+00
25%,1.002903e+00
50%,1.002903e+00
75%,1.002903e+00
max,1.002903e+00


In [ ]:
ad_z.iloc[0, :10]
ad_z.iloc[1, :10]

,A1CF
GSM300166,2.016834
GSM300167,0.208010
GSM300168,-2.682012
GSM300169,-2.934985
GSM300170,-2.792433
GSM300171,-3.506179
GSM300172,-2.396952
GSM300173,0.079872
GSM300174,-0.357549
GSM300175,-0.376347
